# Avoiding Full Scans

## Overview
A **full table scan (Seq Scan)** reads every row in the table regardless of how many rows the query needs. On a 10-million-row table, a seq scan that returns 50 rows is massively wasteful. Understanding which query patterns prevent index use -- and how to rewrite them -- is a core optimisation skill.

**Patterns that prevent index use:**

| Anti-pattern | Why index is skipped | Rewrite |
|---|---|---|
| `WHERE UPPER(col) = 'X'` | Function wraps the column | Normalise data or use functional index |
| `WHERE col::TEXT = '5'` | Cast wraps the column | Use the correct type in the literal |
| `WHERE col LIKE '%suffix'` | Leading wildcard | Full-text search or reverse-string index |
| `WHERE col + 1 = 10` | Arithmetic on column | Rewrite: `WHERE col = 9` |
| `WHERE col != 'X'` | Negation is usually not selective | Rarely fixable; consider partial index |
| `WHERE col IS NOT NULL` | High cardinality | Partial index: `WHERE col IS NOT NULL` |
| `WHERE col IN (subquery)` | Planner may not optimise | Rewrite as JOIN |
| Implicit type mismatch | e.g. integer column vs string literal | Match types explicitly |

**The 5-10% selectivity rule of thumb:**
Indexes are beneficial when a query returns roughly less than 5-10% of the table's rows. If 80% of rows qualify, a seq scan is often faster because reading index entries + heap pages is more I/O than a single sequential table read.

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE sites (site_id INTEGER PRIMARY KEY, site_name TEXT NOT NULL, region TEXT, latitude REAL, longitude REAL, elevation_m REAL, established_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE species (species_id INTEGER PRIMARY KEY, common_name TEXT NOT NULL, scientific_name TEXT NOT NULL UNIQUE, taxon_group TEXT, native INTEGER DEFAULT 1, at_risk INTEGER DEFAULT 0);CREATE TABLE field_crew (crew_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, role TEXT, certified INTEGER DEFAULT 1);CREATE TABLE observations (obs_id INTEGER PRIMARY KEY, site_id INTEGER REFERENCES sites(site_id), species_id INTEGER REFERENCES species(species_id), crew_id INTEGER REFERENCES field_crew(crew_id), obs_date TEXT NOT NULL, count_individuals INTEGER, life_stage TEXT, method TEXT, notes TEXT);CREATE TABLE water_quality (wq_id INTEGER PRIMARY KEY, site_id INTEGER REFERENCES sites(site_id), crew_id INTEGER REFERENCES field_crew(crew_id), sample_date TEXT NOT NULL, ph REAL, dissolved_o2 REAL, turbidity_ntu REAL, temp_c REAL, conductivity_us REAL);INSERT INTO sites VALUES (1,'Fundy Estuary','Atlantic',45.78,-64.52,3.2,'2018-04-01',1),(2,'Kejimkujik Lake','Atlantic',44.43,-65.21,156.0,'2017-06-15',1),(3,'Presqu ile Point','Great Lakes',43.99,-77.72,75.5,'2019-03-20',1),(4,'Rondeau Bay','Great Lakes',42.31,-81.87,176.0,'2018-09-10',1),(5,'Athabasca Delta','Boreal',58.72,-110.87,213.0,'2016-01-15',1),(6,'Wapusk Tundra','Boreal',57.92,-93.15,45.0,'2017-07-01',1),(7,'Clayoquot Sound','Pacific',49.15,-125.93,12.0,'2015-05-20',1),(8,'Boundary Bay','Pacific',49.01,-122.97,1.5,'2016-08-01',1),(9,'Lac Saint-Pierre','St Lawrence',46.19,-72.87,8.0,'2018-02-14',1),(10,'Baie des Chaleurs','Atlantic',48.01,-65.72,5.0,'2019-11-01',0);INSERT INTO species VALUES (1,'Atlantic Salmon','Salmo salar','Fish',1,1),(2,'Great Blue Heron','Ardea herodias','Bird',1,0),(3,'Wood Duck','Aix sponsa','Bird',1,0),(4,'Spotted Turtle','Clemmys guttata','Reptile',1,1),(5,'Common Loon','Gavia immer','Bird',1,0),(6,'Muskrat','Ondatra zibethicus','Mammal',1,0),(7,'Northern Pike','Esox lucius','Fish',1,0),(8,'Bald Eagle','Haliaeetus leucocephalus','Bird',1,0),(9,'American Bittern','Botaurus lentiginosus','Bird',1,1),(10,'Snapping Turtle','Chelydra serpentina','Reptile',1,0),(11,'Rainbow Trout','Oncorhynchus mykiss','Fish',0,0),(12,'Ring-necked Duck','Aythya collaris','Bird',1,0),(13,'Beaver','Castor canadensis','Mammal',1,0),(14,'River Otter','Lontra canadensis','Mammal',1,0),(15,'Painted Turtle','Chrysemys picta','Reptile',1,0);INSERT INTO field_crew VALUES (1,'Dr. Aroha Ngata','Biologist',1),(2,'Liam Chen','Technician',1),(3,'Fatima Al-Rashid','Biologist',1),(4,'James MacLeod','Technician',1),(5,'Sofia Petrov','Student',0);INSERT INTO observations VALUES (1,1,1,1,'2023-04-10',12,'Adult','Electrofishing',NULL),(2,1,2,2,'2023-04-10',3,'Adult','Visual',NULL),(3,2,5,1,'2023-04-15',2,'Adult','Auditory',NULL),(4,2,4,3,'2023-04-15',8,'Juvenile','Visual',NULL),(5,3,2,2,'2023-05-01',5,'Adult','Visual',NULL),(6,3,3,4,'2023-05-01',7,'Adult','Visual',NULL),(7,4,10,3,'2023-05-10',2,'Adult','Visual',NULL),(8,4,7,1,'2023-05-10',4,'Adult','Electrofishing',NULL),(9,5,13,2,'2023-05-20',6,'Adult','Camera Trap',NULL),(10,5,14,3,'2023-05-20',1,'Adult','Visual',NULL),(11,6,8,1,'2023-06-01',3,'Adult','Visual',NULL),(12,6,6,4,'2023-06-01',9,'Adult','Trap',NULL),(13,7,14,3,'2023-06-15',2,'Adult','Visual',NULL),(14,7,5,2,'2023-06-15',4,'Adult','Auditory',NULL),(15,8,2,1,'2023-07-01',12,'Adult','Visual',NULL),(16,8,8,4,'2023-07-01',2,'Adult','Visual',NULL),(17,9,1,3,'2023-07-10',7,'Adult','Electrofishing',NULL),(18,9,9,1,'2023-07-10',1,'Adult','Visual','First sighting this season'),(19,1,4,2,'2023-08-05',1,'Adult','Visual',NULL),(20,2,13,4,'2023-08-05',4,'Adult','Camera Trap',NULL),(21,3,6,3,'2023-08-12',11,'Adult','Trap',NULL),(22,4,2,2,'2023-08-12',7,'Adult','Visual',NULL),(23,5,8,1,'2023-09-01',1,'Adult','Visual',NULL),(24,6,14,3,'2023-09-01',3,'Adult','Visual',NULL),(25,7,1,4,'2023-09-15',18,'Adult','Electrofishing',NULL),(26,8,5,2,'2023-09-15',6,'Adult','Visual',NULL),(27,9,4,3,'2023-09-20',2,'Juvenile','Visual',NULL),(28,1,7,1,'2023-10-01',3,'Adult','Electrofishing',NULL),(29,2,1,2,'2023-10-01',9,'Adult','Electrofishing',NULL),(30,3,8,4,'2023-10-10',4,'Adult','Visual',NULL),(31,4,5,1,'2023-10-15',3,'Adult','Auditory',NULL),(32,5,2,3,'2023-10-15',8,'Adult','Visual',NULL);INSERT INTO water_quality VALUES (1,1,1,'2023-04-10',7.2,9.1,3.4,8.5,142.0),(2,1,2,'2023-06-15',7.4,8.6,4.2,14.2,138.0),(3,1,3,'2023-08-20',7.1,7.8,5.1,19.6,145.0),(4,2,1,'2023-04-15',6.8,10.2,1.2,7.1,98.0),(5,2,4,'2023-07-01',6.9,9.4,1.8,16.3,102.0),(6,3,2,'2023-05-01',7.6,9.8,2.3,11.4,188.0),(7,3,3,'2023-08-01',7.5,8.2,3.7,20.1,192.0),(8,4,1,'2023-05-10',7.8,9.5,1.9,13.0,205.0),(9,4,4,'2023-09-05',7.7,8.9,2.5,18.4,198.0),(10,5,2,'2023-05-20',7.3,10.8,0.8,9.2,55.0),(11,5,1,'2023-09-10',7.2,9.6,1.1,15.7,58.0),(12,6,3,'2023-06-01',6.5,11.2,0.5,6.8,32.0),(13,7,2,'2023-06-15',7.9,9.0,1.4,12.5,310.0),(14,8,4,'2023-07-01',7.8,8.7,2.1,17.2,295.0),(15,9,1,'2023-07-10',7.6,9.3,2.8,14.8,178.0);""")
conn.commit()
print("Ecological schema ready.")

Ecological schema ready.


---
## Function on column prevents index use

In [2]:
conn.executescript("""
    CREATE INDEX idx_obs_method  ON observations (method);
    CREATE INDEX idx_obs_site    ON observations (site_id);
    CREATE INDEX idx_obs_date    ON observations (obs_date);
    CREATE INDEX idx_sp_taxon    ON species (taxon_group);
""")
conn.commit()

print("=== Function on column bypasses index ===")
cases = [
    ("UPPER(method) = 'VISUAL' (function on column -- no index)",
     "SELECT * FROM observations WHERE UPPER(method) = 'VISUAL'"),
    ("method = 'Visual'  (direct equality -- uses index)",
     "SELECT * FROM observations WHERE method = 'Visual'"),
    ("SUBSTR(obs_date,1,7) = '2023-07'  (function on date -- no index)",
     "SELECT * FROM observations WHERE SUBSTR(obs_date,1,7) = '2023-07'"),
    ("obs_date BETWEEN '2023-07-01' AND '2023-07-31'  (range -- uses index)",
     "SELECT * FROM observations WHERE obs_date BETWEEN '2023-07-01' AND '2023-07-31'"),
]
for label, q in cases:
    plan = conn.execute(f"EXPLAIN QUERY PLAN {q}").fetchall()
    uses_idx = any("idx_" in str(r) for r in plan)
    status = "INDEX USED" if uses_idx else "FULL SCAN"
    print(f"  [{status}]  {label}")


=== Function on column bypasses index ===
  [FULL SCAN]  UPPER(method) = 'VISUAL' (function on column -- no index)
  [INDEX USED]  method = 'Visual'  (direct equality -- uses index)
  [FULL SCAN]  SUBSTR(obs_date,1,7) = '2023-07'  (function on date -- no index)
  [INDEX USED]  obs_date BETWEEN '2023-07-01' AND '2023-07-31'  (range -- uses index)


---
## Arithmetic on column and leading wildcards

In [3]:
print("=== Arithmetic on column prevents index use ===")
conn.execute("CREATE INDEX idx_obs_count ON observations (count_individuals)")
conn.commit()

arith_cases = [
    ("count_individuals * 2 > 10  (arithmetic on column)",
     "SELECT * FROM observations WHERE count_individuals * 2 > 10"),
    ("count_individuals > 5  (direct comparison -- uses index)",
     "SELECT * FROM observations WHERE count_individuals > 5"),
    ("LIKE '%fishing'  (leading wildcard -- no index)",
     "SELECT * FROM observations WHERE method LIKE '%fishing'"),
    ("LIKE 'Electro%'  (trailing wildcard -- uses index)",
     "SELECT * FROM observations WHERE method LIKE 'Electro%'"),
]
for label, q in arith_cases:
    plan = conn.execute(f"EXPLAIN QUERY PLAN {q}").fetchall()
    uses_idx = any("idx_" in str(r) for r in plan)
    status = "INDEX USED" if uses_idx else "FULL SCAN"
    print(f"  [{status}]  {label}")

print()
print("PostgreSQL functional index workaround for UPPER():")
print("""
  -- Create a functional index so UPPER(common_name) queries use an index:
  CREATE INDEX idx_species_name_upper ON species (UPPER(common_name));

  -- Now this uses the index:
  SELECT * FROM species WHERE UPPER(common_name) = 'ATLANTIC SALMON';

  -- Equivalent: normalise at insert time (store names in lowercase/uppercase)
  -- so no function is needed at query time.
""")


=== Arithmetic on column prevents index use ===
  [FULL SCAN]  count_individuals * 2 > 10  (arithmetic on column)
  [INDEX USED]  count_individuals > 5  (direct comparison -- uses index)
  [FULL SCAN]  LIKE '%fishing'  (leading wildcard -- no index)
  [FULL SCAN]  LIKE 'Electro%'  (trailing wildcard -- uses index)

PostgreSQL functional index workaround for UPPER():

  -- Create a functional index so UPPER(common_name) queries use an index:
  CREATE INDEX idx_species_name_upper ON species (UPPER(common_name));

  -- Now this uses the index:
  SELECT * FROM species WHERE UPPER(common_name) = 'ATLANTIC SALMON';

  -- Equivalent: normalise at insert time (store names in lowercase/uppercase)
  -- so no function is needed at query time.



---
## IN with subquery vs JOIN, and OR rewrites

In [4]:
print("=== IN (subquery) vs JOIN ===")
q_in = """
EXPLAIN QUERY PLAN
SELECT obs_id, obs_date FROM observations
WHERE  site_id IN (SELECT site_id FROM sites WHERE region = 'Atlantic')
"""
q_join = """
EXPLAIN QUERY PLAN
SELECT o.obs_id, o.obs_date FROM observations o
JOIN   sites s ON o.site_id = s.site_id
WHERE  s.region = 'Atlantic'
"""
plan_in   = conn.execute(q_in).fetchall()
plan_join = conn.execute(q_join).fetchall()
print("IN (subquery) plan:")
for r in plan_in:   print(" ", r)
print("JOIN plan:")
for r in plan_join: print(" ", r)

print()
print("=== OR on different columns prevents composite index use ===")
print("""Problematic: WHERE site_id = 1 OR obs_date = '2023-07-10'
  Each condition uses a different index column.
  The planner typically does two index scans and unions results (Bitmap Or).

Rewrite options:
  1. UNION ALL (if values are known):
     SELECT * FROM observations WHERE site_id = 1
     UNION ALL
     SELECT * FROM observations WHERE obs_date = '2023-07-10'
       AND site_id != 1;   -- avoid double-counting

  2. Accept the Bitmap Or plan -- it is often fast enough.

  3. Denormalise the OR into separate queries in application code.
""")
conn.close()


=== IN (subquery) vs JOIN ===
IN (subquery) plan:
  (3, 0, 0, 'SEARCH observations USING INDEX idx_obs_site (site_id=?)')
  (7, 0, 0, 'LIST SUBQUERY 1')
  (9, 7, 0, 'SCAN sites')
JOIN plan:
  (4, 0, 0, 'SCAN s')
  (8, 0, 0, 'SEARCH o USING INDEX idx_obs_site (site_id=?)')

=== OR on different columns prevents composite index use ===
Problematic: WHERE site_id = 1 OR obs_date = '2023-07-10'
  Each condition uses a different index column.
  The planner typically does two index scans and unions results (Bitmap Or).

Rewrite options:
  1. UNION ALL (if values are known):
     SELECT * FROM observations WHERE site_id = 1
     UNION ALL
     SELECT * FROM observations WHERE obs_date = '2023-07-10'
       AND site_id != 1;   -- avoid double-counting

  2. Accept the Bitmap Or plan -- it is often fast enough.

  3. Denormalise the OR into separate queries in application code.



---
## Common Pitfalls

**1. Wrapping columns in functions in WHERE clauses**
`WHERE DATE_TRUNC('month', obs_date) = '2023-07-01'` wraps the column in a function, bypassing the index on `obs_date`. Rewrite as `WHERE obs_date >= '2023-07-01' AND obs_date < '2023-08-01'` to keep the column bare and allow index range scans.

**2. Leading wildcard LIKE patterns**
`WHERE method LIKE '%fishing'` cannot use a B-tree index because the match point is at the end of the string. If suffix searches are frequent, store the reversed string and use a trailing wildcard, or use a full-text GIN index in PostgreSQL: `WHERE method_tsvector @@ to_tsquery('fishing')`.

**3. Implicit type coercion between column and literal**
`WHERE site_id = '1'` (string literal vs integer column) may coerce the column to text for comparison in some databases, bypassing the integer index. Always match the literal type to the column type: `WHERE site_id = 1`.

**4. NOT IN with a subquery that can return NULL**
`WHERE species_id NOT IN (SELECT species_id FROM some_table)` returns zero rows if any row in the subquery has a NULL species_id -- because `x NOT IN (1, 2, NULL)` evaluates to UNKNOWN (not TRUE) for all x. Use `NOT EXISTS` or `LEFT JOIN ... WHERE rhs.key IS NULL` instead.

**5. Using OR instead of UNION ALL for different columns**
`WHERE site_id = 1 OR species_id = 5` requires the planner to satisfy two different index conditions. It typically uses a Bitmap Or of two separate index scans. For large tables, UNION ALL with two targeted queries (each using its own index) is often faster.

**6. Negation predicates (!=, NOT LIKE, NOT IN) are rarely index-friendly**
`WHERE region != 'Atlantic'` returns ~80% of rows -- a seq scan is faster. Negation indexes exist but are generally not useful because high row counts make index traversal more expensive than a sequential read. Design queries to use positive filters where possible.


---
*sql_methods_library - Samantha McGarrigle*